In [27]:
import json

with open(
    "../data/candidate_profile.json",
    "r",
    encoding="utf-8"
) as f:

    candidate = json.load(f)

In [28]:
candidate.keys()

dict_keys(['name', 'email', 'phone', 'education', 'skills', 'projects', 'experience', 'certifications', 'achievements'])

In [29]:
target_role = "Data Scientist"

print("Target role:", target_role)

Target role: Data Scientist


In [30]:
ROLE_PROFILES = {
    "data scientist": {
        "technical_skills": [
            "Python",
            "SQL",
            "Pandas",
            "NumPy",
            "Scikit-learn"
        ],

        "core_concepts": [
            "Statistics",
            "Probability",
            "Machine Learning",
            "Feature Engineering",
            "Model Evaluation",
            "Data Analysis"
        ],

        "advanced_concepts": [
            "Ensemble Methods",
            "Gradient Boosting",
            "XGBoost",
            "Cross Validation",
            "Hyperparameter Tuning",
            "A/B Testing"
        ],

        "problem_solving": [
            "Problem formulation",
            "Model selection",
            "Error analysis",
            "Trade-off reasoning"
        ],

        "communication": [
            "Explaining technical decisions",
            "Interpreting results",
            "Business reasoning"
        ]
    }
}

In [31]:
role = ROLE_PROFILES["data scientist"]

role

{'technical_skills': ['Python', 'SQL', 'Pandas', 'NumPy', 'Scikit-learn'],
 'core_concepts': ['Statistics',
  'Probability',
  'Machine Learning',
  'Feature Engineering',
  'Model Evaluation',
  'Data Analysis'],
 'advanced_concepts': ['Ensemble Methods',
  'Gradient Boosting',
  'XGBoost',
  'Cross Validation',
  'Hyperparameter Tuning',
  'A/B Testing'],
 'problem_solving': ['Problem formulation',
  'Model selection',
  'Error analysis',
  'Trade-off reasoning'],
 'communication': ['Explaining technical decisions',
  'Interpreting results',
  'Business reasoning']}

In [32]:
candidate_skills = {
    skill.lower()
    for skill in candidate["skills"]
}

role_skills = {
    skill.lower()
    for skill in role["technical_skills"]
}

matched_skills = candidate_skills.intersection(
    role_skills
)

missing_skills = role_skills - candidate_skills

In [33]:
print("MATCHED:")
print(matched_skills)

print("\nMISSING:")
print(missing_skills)

MATCHED:
{'scikit-learn', 'python', 'pandas', 'numpy', 'sql'}

MISSING:
set()


In [34]:
competencies = []

for category, items in role.items():

    for item in items:

        competencies.append({
            "name": item,
            "category": category
        })

In [35]:
competencies[:10]

[{'name': 'Python', 'category': 'technical_skills'},
 {'name': 'SQL', 'category': 'technical_skills'},
 {'name': 'Pandas', 'category': 'technical_skills'},
 {'name': 'NumPy', 'category': 'technical_skills'},
 {'name': 'Scikit-learn', 'category': 'technical_skills'},
 {'name': 'Statistics', 'category': 'core_concepts'},
 {'name': 'Probability', 'category': 'core_concepts'},
 {'name': 'Machine Learning', 'category': 'core_concepts'},
 {'name': 'Feature Engineering', 'category': 'core_concepts'},
 {'name': 'Model Evaluation', 'category': 'core_concepts'}]

In [36]:
candidate_context = {
    "skills": candidate["skills"],
    "projects": candidate["projects"],
    "experience": candidate["experience"]
}

In [37]:
matched_count = len(matched_skills)
total_role_skills = len(role_skills)

role_match_score = (
    matched_count / total_role_skills
) * 100

print(
    f"Technical skill match: "
    f"{role_match_score:.1f}%"
)

Technical skill match: 100.0%


In [38]:
interview_blueprint = {
    "target_role": target_role,

    "competencies": competencies,

    "candidate_strengths": list(matched_skills),

    "candidate_gaps": list(missing_skills),

    "priority_topics": list(missing_skills),

    "difficulty": "medium"
}

In [39]:
interview_blueprint

{'target_role': 'Data Scientist',
 'competencies': [{'name': 'Python', 'category': 'technical_skills'},
  {'name': 'SQL', 'category': 'technical_skills'},
  {'name': 'Pandas', 'category': 'technical_skills'},
  {'name': 'NumPy', 'category': 'technical_skills'},
  {'name': 'Scikit-learn', 'category': 'technical_skills'},
  {'name': 'Statistics', 'category': 'core_concepts'},
  {'name': 'Probability', 'category': 'core_concepts'},
  {'name': 'Machine Learning', 'category': 'core_concepts'},
  {'name': 'Feature Engineering', 'category': 'core_concepts'},
  {'name': 'Model Evaluation', 'category': 'core_concepts'},
  {'name': 'Data Analysis', 'category': 'core_concepts'},
  {'name': 'Ensemble Methods', 'category': 'advanced_concepts'},
  {'name': 'Gradient Boosting', 'category': 'advanced_concepts'},
  {'name': 'XGBoost', 'category': 'advanced_concepts'},
  {'name': 'Cross Validation', 'category': 'advanced_concepts'},
  {'name': 'Hyperparameter Tuning', 'category': 'advanced_concepts'},
 

In [40]:
print("Role:", target_role)
print("Matched skills:", matched_skills)
print("Missing skills:", missing_skills)
print("Match score:", role_match_score)
print("Competencies:", len(competencies))

Role: Data Scientist
Matched skills: {'scikit-learn', 'python', 'pandas', 'numpy', 'sql'}
Missing skills: set()
Match score: 100.0
Competencies: 24


# LLM role analysis

In [42]:
from pydantic import BaseModel, Field

In [44]:
class CompetencyAssessment(BaseModel):
    competency: str
    category: str
    status: str
    evidence: str = ""
    priority: str
    
class RoleAnalysis(BaseModel):
    target_role: str
    competencies: list[CompetencyAssessment] = Field(
        default_factory=list
    )
    strengths: list[str] = Field(default_factory=list)
    unverified_areas: list[str] = Field(default_factory=list)
    priority_topics: list[str] = Field(default_factory=list)
    recommended_difficulty: str = "medium"

In [45]:
ROLE_ANALYSIS_PROMPT = """
You are an AI interview planning system.

Your task is to analyze a candidate against a target job role.

You are given:

1. The target role
2. A baseline competency map for that role
3. The candidate's resume-derived profile

Your job is to determine:

- Which competencies are clearly demonstrated
- Which are partially demonstrated
- Which are unverified
- What evidence exists in the candidate profile
- Which topics should receive interview priority
- What difficulty level is appropriate

Rules:

1. Never claim that a candidate is weak merely because a skill is absent from the resume.
2. Use "UNVERIFIED" when the resume provides insufficient evidence.
3. Only use evidence explicitly present in the candidate profile.
4. Do not invent candidate experience.
5. Prioritize competencies that are important for the role but insufficiently demonstrated.
6. Consider the candidate's projects and experience, not only their skills list.
7. The goal is to design an interview, not to make a final hiring decision.
8. Return only structured JSON matching the provided schema.
"""

In [46]:
import json

role_analysis_input = {
    "target_role": target_role,
    "role_profile": role,
    "candidate_profile": candidate
}

In [47]:
role_analysis_input_text = json.dumps(
    role_analysis_input,
    indent=2,
    ensure_ascii=False
)

print(role_analysis_input_text)

{
  "target_role": "Data Scientist",
  "role_profile": {
    "technical_skills": [
      "Python",
      "SQL",
      "Pandas",
      "NumPy",
      "Scikit-learn"
    ],
    "core_concepts": [
      "Statistics",
      "Probability",
      "Machine Learning",
      "Feature Engineering",
      "Model Evaluation",
      "Data Analysis"
    ],
    "advanced_concepts": [
      "Ensemble Methods",
      "Gradient Boosting",
      "XGBoost",
      "Cross Validation",
      "Hyperparameter Tuning",
      "A/B Testing"
    ],
    "problem_solving": [
      "Problem formulation",
      "Model selection",
      "Error analysis",
      "Trade-off reasoning"
    ],
    "communication": [
      "Explaining technical decisions",
      "Interpreting results",
      "Business reasoning"
    ]
  },
  "candidate_profile": {
    "name": "Bhumi Saraogi",
    "email": "saraogibhumi@gmail.com",
    "phone": "(+91) 9531657378",
    "education": [
      {
        "degree": "B.Tech. in Computer Science and E

In [51]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env")

groq_api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=groq_api_key)

In [52]:
schema = RoleAnalysis.model_json_schema()

In [53]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": ROLE_ANALYSIS_PROMPT
        },
        {
            "role": "user",
            "content": f"""
Target role and candidate information:

{role_analysis_input_text}

Required JSON schema:

{json.dumps(schema, indent=2)}
"""
        }
    ],
    temperature=0
)

In [54]:
response_text = response.choices[0].message.content

print(response_text)

{
  "target_role": "Data Scientist",
  "competencies": [
    {
      "competency": "Python",
      "category": "Technical Skills",
      "status": "Demonstrated",
      "evidence": "Listed in skills; used in all projects (SceneSense AI, SnapClass, Anonify).",
      "priority": "Low"
    },
    {
      "competency": "SQL",
      "category": "Technical Skills",
      "status": "Demonstrated",
      "evidence": "Listed in skills; used for data storage/querying in SnapClass (Supabase).",
      "priority": "Low"
    },
    {
      "competency": "Pandas",
      "category": "Technical Skills",
      "status": "Demonstrated",
      "evidence": "Listed in skills; typical for data manipulation in Python projects.",
      "priority": "Low"
    },
    {
      "competency": "NumPy",
      "category": "Technical Skills",
      "status": "Demonstrated",
      "evidence": "Listed in skills; foundational library for numerical work in ML projects.",
      "priority": "Low"
    },
    {
      "competency

In [55]:
parsed_analysis = json.loads(response_text)

In [56]:
role_analysis = RoleAnalysis.model_validate(
    parsed_analysis
)

In [57]:
role_analysis

RoleAnalysis(target_role='Data Scientist', competencies=[CompetencyAssessment(competency='Python', category='Technical Skills', status='Demonstrated', evidence='Listed in skills; used in all projects (SceneSense AI, SnapClass, Anonify).', priority='Low'), CompetencyAssessment(competency='SQL', category='Technical Skills', status='Demonstrated', evidence='Listed in skills; used for data storage/querying in SnapClass (Supabase).', priority='Low'), CompetencyAssessment(competency='Pandas', category='Technical Skills', status='Demonstrated', evidence='Listed in skills; typical for data manipulation in Python projects.', priority='Low'), CompetencyAssessment(competency='NumPy', category='Technical Skills', status='Demonstrated', evidence='Listed in skills; foundational library for numerical work in ML projects.', priority='Low'), CompetencyAssessment(competency='Scikit-learn', category='Technical Skills', status='Demonstrated', evidence='Listed in skills; likely used for model pipelines in 

In [58]:
for competency in role_analysis.competencies:

    print(
        competency.competency,
        "|",
        competency.status,
        "|",
        competency.priority
    )

Python | Demonstrated | Low
SQL | Demonstrated | Low
Pandas | Demonstrated | Low
NumPy | Demonstrated | Low
Scikit-learn | Demonstrated | Low
Statistics | UNVERIFIED | High
Probability | UNVERIFIED | High
Machine Learning | Partially Demonstrated | Medium
Feature Engineering | UNVERIFIED | High
Model Evaluation | UNVERIFIED | High
Data Analysis | UNVERIFIED | High
Ensemble Methods | UNVERIFIED | High
Gradient Boosting | UNVERIFIED | High
XGBoost | UNVERIFIED | High
Cross Validation | UNVERIFIED | High
Hyperparameter Tuning | UNVERIFIED | High
A/B Testing | UNVERIFIED | High
Problem formulation | Partially Demonstrated | Medium
Model selection | Partially Demonstrated | Medium
Error analysis | UNVERIFIED | High
Trade-off reasoning | UNVERIFIED | High
Explaining technical decisions | UNVERIFIED | High
Interpreting results | UNVERIFIED | High
Business reasoning | UNVERIFIED | High


In [61]:
print("STRENGTHS")

for item in role_analysis.strengths:
    print("-", item)
    
print("UNVERIFIED")

for item in role_analysis.unverified_areas:
    print("-", item)
    
print("PRIORITY TOPICS")

for item in role_analysis.priority_topics:
    print("-", item)

STRENGTHS
- Python
- SQL
- Pandas
- NumPy
- Scikit-learn
UNVERIFIED
- Statistics
- Probability
- Feature Engineering
- Model Evaluation
- Data Analysis
- Ensemble Methods
- Gradient Boosting
- XGBoost
- Cross Validation
- Hyperparameter Tuning
- A/B Testing
- Error analysis
- Trade-off reasoning
- Explaining technical decisions
- Interpreting results
- Business reasoning
PRIORITY TOPICS
- Statistics
- Probability
- Feature Engineering
- Model Evaluation
- Data Analysis
- Ensemble Methods
- Gradient Boosting
- XGBoost
- Cross Validation
- Hyperparameter Tuning
- A/B Testing
- Error analysis
- Trade-off reasoning
- Explaining technical decisions
- Interpreting results
- Business reasoning
- Machine Learning
- Problem formulation
- Model selection


In [62]:
print(
    "Recommended difficulty:",
    role_analysis.recommended_difficulty
)

Recommended difficulty: medium


In [63]:
interview_blueprint = {
    "target_role": role_analysis.target_role,
    "competencies": [
        item.model_dump()
        for item in role_analysis.competencies
    ],
    "strengths": role_analysis.strengths,
    "unverified_areas": role_analysis.unverified_areas,
    "priority_topics": role_analysis.priority_topics,
    "recommended_difficulty": role_analysis.recommended_difficulty
}

In [64]:
with open(
    "../data/interview_blueprint.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        interview_blueprint,
        f,
        indent=2,
        ensure_ascii=False
    )